# Approach 2 / NB2 — Fine-tune Whisper to output phonemes (ANPLers method)

Fine-tunes a Whisper model so it transcribes audio into a **phoneme sequence**
(e.g. `f ii h i nn a ...`) instead of normal Arabic text. Keeping the seq2seq
decoder, we train on (audio -> phoneme_ref string). Because the phonemes are
plain space-separated symbols, Whisper's existing tokenizer encodes them directly.

**Base model choices (set MODEL_NAME):**
- `tarteel-ai/whisper-base-ar-quran` — Quran-tuned encoder, fits T4 easily (recommended start).
- `openai/whisper-small` / `openai/whisper-medium` — more capacity, slower.
- `openai/whisper-large-v3` — best per the paper, but needs A100-class GPU.

**Hardware:** base/small fit Kaggle T4. fp16.

> NOTE (honest expectation): Approach 2 is the SECONDARY route. On the IqraEval
> benchmark, encoder-decoder Whisper-as-phoneme scored ~F1 0.32 (lower than the
> wav2vec2-BERT CTC route in Approach 1). Use this as a comparison/fallback.

In [1]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" "accelerate==0.34.2" jiwer soundfile librosa
import warnings
warnings.filterwarnings("ignore", message=".*use_reentrant.*")
warnings.filterwarnings("ignore", category=UserWarning)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires googl

In [ ]:
import os, time
import numpy as np
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_dataset, load_from_disk, Audio, concatenate_datasets

# ---- Single-GPU guard ---------------------------------------------------
# On Kaggle 2x T4 the Trainer uses nn.DataParallel, which gathers each GPU's
# outputs; variable-length generation/CTC outputs differ in shape and gather
# crashes. Restrict to one GPU (DataParallel only activates with 2+).
if torch.cuda.device_count() > 1:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print("Multiple GPUs -> restricting to GPU 0 (avoids DataParallel gather bug).")
print("visible GPUs:", torch.cuda.device_count())

# ---- HF auth + robust download ------------------------------------------
try:
    from kaggle_secrets import UserSecretsClient
    _t = UserSecretsClient().get_secret("HF_TOKEN"); os.environ["HF_TOKEN"] = _t
    os.environ["HUGGING_FACE_HUB_TOKEN"] = _t; print("HF_TOKEN loaded.")
except Exception as e:
    print("No HF_TOKEN (anonymous):", e)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")

def robust_load_dataset(repo, split, retries=6, wait=20):
    for i in range(retries):
        try:
            return load_dataset(repo, split=split)
        except Exception as e:
            print(f"  load {repo}[{split}] try {i+1}/{retries}: {type(e).__name__}: {str(e)[:140]}")
            if i == retries - 1: raise
            time.sleep(wait * (i + 1))

# ---- CONFIG -------------------------------------------------------------
MODEL_NAME = "tarteel-ai/whisper-base-ar-quran"   # or openai/whisper-small, etc.
SAMPLING_RATE = 16_000
TRAIN_REPO = "IqraEval/Iqra_train"
TTS_REPO   = "IqraEval/Iqra_TTS"
USE_TTS    = True
PHONEME_COL = "phoneme_ref"
MAX_LABEL_LEN = 448

# RECOMMENDED: attach the cached dataset (from NB1 or Approach-1 iqra-cache) to skip downloads.
LOCAL_TRAIN_DISK = "/kaggle/input/datasets/mabubakar29/model1-dataset/iqra_prep/iqra_train_arrow"  # e.g. "/kaggle/input/iqra-cache/iqra_train_arrow"
LOCAL_TTS_DISK   = "/kaggle/input/datasets/mabubakar29/model1-dataset/iqra_prep/iqra_tts_arrow"  # e.g. "/kaggle/input/iqra-cache/iqra_tts_arrow"

OUTPUT_DIR = "/kaggle/working/whisper-quran-phoneme"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- SMOKE TEST ---------------------------------------------------------
# Smoke run passed (PER 0.58 -> 0.38 in 200 steps). For the real comparison vs
# Approach 1 (wav2vec2, final PER 0.071), do the FULL run: SMOKE_TEST = False.
# 5000 steps (NOT 15000) — Approach 2 is the secondary/comparison route, and 5k
# already gives a representative PER while staying safely under Kaggle's time +
# disk limits (the same limits the A1 run hit at ~4h).
SMOKE_TEST = False
if SMOKE_TEST:
    USE_TTS = False; MAX_STEPS = 200; EVAL_STEPS = 100; SUBSET_TRAIN = 512; SUBSET_DEV = 128
else:
    MAX_STEPS = 5000; EVAL_STEPS = 1000; SUBSET_TRAIN = None; SUBSET_DEV = None

print("MODEL:", MODEL_NAME, "| SMOKE_TEST:", SMOKE_TEST, "| USE_TTS:", USE_TTS, "| MAX_STEPS:", MAX_STEPS)

In [3]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Whisper processor: feature extractor (audio->log-mel) + tokenizer (text<->ids).
# NOTE: we do NOT pass language/task here. We're training the model to emit a
# phoneme string, not language-tagged Arabic, and Tarteel's base config lacks
# the lang_to_id table that language forcing needs (see cell 8).
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
print("processor ready. Example phoneme tokenization:")
demo = "f ii h i nn a x A y r aa t"
ids = processor.tokenizer(demo).input_ids
print("  ids:", ids)
print("  back:", processor.tokenizer.decode(ids, skip_special_tokens=True))

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/827 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


processor ready. Example phoneme tokenization:
  ids: [50258, 50363, 69, 741, 72, 276, 741, 297, 77, 257, 2031, 316, 288, 367, 40079, 220, 83, 50257]


2026-06-27 09:49:55.982142: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782553796.386896      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782553796.501461      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782553797.609378      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782553797.609436      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782553797.609439      23 computation_placer.cc:177] computation placer alr

  back: f ii h i nn a x A y r aa t


In [ ]:
# ---- Load + merge data --------------------------------------------------
# Iqra_train audio is CLEAN  -> label = phoneme_ref.
# Iqra_TTS audio is MISPRONOUNCED -> label = phoneme_mis (matches the audio!).
if LOCAL_TRAIN_DISK:
    full = load_from_disk(LOCAL_TRAIN_DISK); train, dev = full["train"], full["dev"]
    print("loaded train+dev from cache")
else:
    train = robust_load_dataset(TRAIN_REPO, "train")
    dev   = robust_load_dataset(TRAIN_REPO, "dev")

def normalise(ds, label_col):
    keep = {"audio", label_col}
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep])
    if label_col != "phon": ds = ds.rename_column(label_col, "phon")
    return ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

train = normalise(train, PHONEME_COL); dev = normalise(dev, PHONEME_COL)

if USE_TTS:
    try:
        if LOCAL_TTS_DISK:
            tf = load_from_disk(LOCAL_TTS_DISK)
            tts = tf["train"] if "train" in getattr(tf, "keys", lambda: [])() else tf
        else:
            tts = robust_load_dataset(TTS_REPO, "train")
        tcol = "phoneme_mis" if "phoneme_mis" in tts.column_names else PHONEME_COL
        print("TTS label column:", tcol)
        tts = normalise(tts, tcol)
        train = concatenate_datasets([train, tts])
    except Exception as e:
        print("TTS skipped:", e)

# keep_in_memory=True: data loaded from /kaggle/input is read-only, and .shuffle()
# would otherwise try to write its index temp file there -> [Errno 30]. In-memory
# shuffle avoids that.
if SUBSET_TRAIN: train = train.shuffle(seed=0, keep_in_memory=True).select(range(min(SUBSET_TRAIN, len(train))))
if SUBSET_DEV:   dev   = dev.shuffle(seed=0, keep_in_memory=True).select(range(min(SUBSET_DEV, len(dev))))

# EVAL SPEED: Whisper eval uses predict_with_generate, which is SLOW (~38s/128
# clips). The full dev set is 2588 clips -> ~13 min PER eval x5 = ~1h wasted.
# Cap the eval set to a fixed sample so each eval is quick; it's plenty to track
# PER for a comparison run. (Train set is untouched.)
EVAL_CAP = 400
if not SMOKE_TEST and len(dev) > EVAL_CAP:
    dev = dev.shuffle(seed=0, keep_in_memory=True).select(range(EVAL_CAP))

print("train:", len(train), "| dev:", len(dev))

In [5]:
# ---- Lazy preprocessing: audio -> log-mel, phoneme string -> label ids ---
def make_transform(b):
    feats = processor.feature_extractor(
        [a["array"] for a in b["audio"]], sampling_rate=SAMPLING_RATE)
    labels = [processor.tokenizer(p).input_ids for p in b["phon"]]
    return {"input_features": feats.input_features, "labels": labels}

train_proc = train.with_transform(make_transform)
dev_proc   = dev.with_transform(make_transform)
print("lazy transforms ready")

lazy transforms ready


In [6]:
@dataclass
class WhisperCollator:
    processor: Any
    def __call__(self, features):
        feat = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(feat, return_tensors="pt")
        labs = [{"input_ids": f["labels"]} for f in features]
        labs_batch = self.processor.tokenizer.pad(labs, return_tensors="pt")
        labels = labs_batch["input_ids"].masked_fill(labs_batch.attention_mask.ne(1), -100)
        # Drop the BOS the model prepends itself.
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

collator = WhisperCollator(processor)

In [7]:
import jiwer

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids = np.where(label_ids == -100, processor.tokenizer.pad_token_id, label_ids)
    pred_str  = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    # PER = error rate over space-separated phoneme tokens.
    per = jiwer.wer(label_str, pred_str)
    return {"per": per}

In [8]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# We are training Whisper to emit a PHONEME sequence, not language-tagged text.
# Tarteel's whisper-base config lacks the multilingual `lang_to_id` table, so
# forcing language="ar" at generation time crashes with:
#   AttributeError: 'GenerationConfig' object has no attribute 'lang_to_id'
# Fix: don't force a language/task. Clear them everywhere and let the model
# generate freely from the audio. (We also clear forced_decoder_ids.)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = None
model.config.use_cache = False
model.generation_config.forced_decoder_ids = None
model.generation_config.language = None
model.generation_config.task = None
# Belt-and-suspenders: ensure no language tokens are forced.
if hasattr(model.generation_config, "suppress_tokens"):
    model.generation_config.suppress_tokens = None

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=0 if SMOKE_TEST else 500,
    max_steps=MAX_STEPS,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # silences per-step warning spam
    predict_with_generate=True,
    generation_max_length=MAX_LABEL_LEN,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=EVAL_STEPS,
    # DISK SPACE: Kaggle /kaggle/working is ~20 GB; checkpoints (weights +
    # optimizer state) are several GB each. Keep only the best one.
    save_total_limit=1,
    save_safetensors=True,
    logging_steps=100 if SMOKE_TEST else 200,
    disable_tqdm=True,
    load_best_model_at_end=not SMOKE_TEST,
    metric_for_best_model="per",
    greater_is_better=False,
    # REQUIRED: we use .with_transform, so input_features/labels are produced
    # lazily and don't appear in dataset.column_names. Without this the Trainer
    # strips every column -> "No columns match the model's forward signature".
    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none",
)

cbs = [] if SMOKE_TEST else [EarlyStoppingCallback(early_stopping_patience=3)]
kw = dict(args=args, model=model, train_dataset=train_proc, eval_dataset=dev_proc,
          data_collator=collator, compute_metrics=compute_metrics, callbacks=cbs)
try:
    trainer = Seq2SeqTrainer(processing_class=processor.feature_extractor, **kw)
except TypeError:
    trainer = Seq2SeqTrainer(tokenizer=processor.feature_extractor, **kw)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/290M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


In [9]:
trainer.train()
trainer.save_model(os.path.join(OUTPUT_DIR, "final"))
processor.save_pretrained(os.path.join(OUTPUT_DIR, "final"))
print("saved ->", os.path.join(OUTPUT_DIR, "final"))

{'loss': 3.1495, 'grad_norm': 9.919824600219727, 'learning_rate': 5.150000000000001e-06, 'epoch': 3.125}


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 1024, 'begin_suppress_tokens': [220, 50257]}


{'eval_loss': 1.5108752250671387, 'eval_per': 0.5815528921313184, 'eval_runtime': 38.2988, 'eval_samples_per_second': 3.342, 'eval_steps_per_second': 0.418, 'epoch': 3.125}
{'loss': 1.152, 'grad_norm': 9.686040878295898, 'learning_rate': 1.5000000000000002e-07, 'epoch': 6.25}


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 1024, 'begin_suppress_tokens': [220, 50257]}


{'eval_loss': 1.1166155338287354, 'eval_per': 0.37597707139134967, 'eval_runtime': 34.5244, 'eval_samples_per_second': 3.708, 'eval_steps_per_second': 0.463, 'epoch': 6.25}


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 1024, 'begin_suppress_tokens': [220, 50257]}


{'train_runtime': 291.5184, 'train_samples_per_second': 10.977, 'train_steps_per_second': 0.686, 'train_loss': 2.1507420349121094, 'epoch': 6.25}
saved -> /kaggle/working/whisper-quran-phoneme/final


### Output
- `whisper-quran-phoneme/final/` — fine-tuned model + processor. Use in NB3.

Smoke test first (`SMOKE_TEST=True`): confirm loss drops + PER prints. Then set
`SMOKE_TEST=False`, attach the cached dataset (`LOCAL_*_DISK`), and Commit the full run.